# 09 — Joint Inference & Evaluation (v4)

**Goal:** Evaluate and visualize the joint vehicle detection + lane segmentation model.

Features:
- Qualitative inference with detection boxes + lane overlay
- Full validation set evaluation (mAP + lane IoU/F1)
- Per-class AP breakdown for vehicle classes
- Lane threshold sensitivity analysis

## 1 — Setup

In [ ]:
!pip install -q ultralytics>=8.4.0 opencv-python matplotlib pyyaml
!pip install -q "torchmetrics[detection]"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, random
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt

ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
DATASET_DIR = "/content/bdd100k_yolo"
JOINT_WEIGHTS = f"{ECOCAR_ROOT}/weights/vehicle_lane_v4_best.pt"

PROJECT_DIR = os.path.join(ECOCAR_ROOT, "project")
assert os.path.isdir(PROJECT_DIR), f"Project not found at {PROJECT_DIR}"
sys.path.insert(0, PROJECT_DIR)

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Joint weights: {JOINT_WEIGHTS}")

## 2 — Load Model

In [ ]:
from src.multitask_model import build_multitask_model
from src.utils.class_map import VEHICLE_CLASSES, NUM_VEHICLE_CLASSES

assert os.path.isfile(JOINT_WEIGHTS), f"Checkpoint not found: {JOINT_WEIGHTS}"

ckpt = torch.load(JOINT_WEIGHTS, map_location='cuda', weights_only=False)
arch = ckpt.get('arch_config', {})
epoch = ckpt.get('epoch', '?')
print(f"Checkpoint epoch: {epoch}")
if 'metrics' in ckpt:
    for k, v in ckpt['metrics'].items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

model = build_multitask_model(
    weights='yolo26s.pt',
    lane_head_type=arch.get('lane_head_type', 'transformer'),
    lane_embed_dim=arch.get('lane_embed_dim', 128),
    lane_num_heads=arch.get('lane_num_heads', 4),
    lane_depth=arch.get('lane_depth', 2),
    mask_height=arch.get('mask_height', 160),
    mask_width=arch.get('mask_width', 160),
)

missing, unexpected = model.load_state_dict(ckpt['model_state_dict'], strict=False)
print(f"Missing keys: {len(missing)}, Unexpected keys: {len(unexpected)}")

if 'ema_state_dict' in ckpt:
    from src.trainers.ema import ModelEMA
    ema = ModelEMA(model, decay=0.9999)
    ema.load_state_dict(ckpt['ema_state_dict'])
    ema.apply(model)
    print("Loaded EMA weights")

model = model.to('cuda').eval()
print("Model loaded.")

## 3 — Qualitative Inference

In [ ]:
BOX_COLORS = [
    (60, 180, 255),   # car - blue
    (100, 100, 255),  # truck - purple
    (255, 220, 60),   # bus - yellow
    (100, 255, 100),  # motorcycle - green
    (255, 100, 200),  # bicycle - pink
]

try:
    from ultralytics.utils.nms import non_max_suppression
except ImportError:
    from ultralytics.utils.ops import non_max_suppression

IMG_SIZE = 640
CONF_THRESH = 0.3
LANE_THRESH = 0.5

def run_joint_inference(model, img_bgr, device='cuda'):
    orig_h, orig_w = img_bgr.shape[:2]
    img_resized = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE))
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(img_rgb.astype(np.float32) / 255.0)
    tensor = tensor.permute(2, 0, 1).unsqueeze(0).to(device)

    with torch.no_grad(), torch.amp.autocast('cuda'):
        out = model(tensor)

    det_out = out['det_output']

    # Handle YOLO26 end2end output: eval returns (y_postprocessed, preds_dict)
    if isinstance(det_out, (tuple, list)) and len(det_out) == 2:
        y_post = det_out[0]
        if isinstance(y_post, torch.Tensor) and y_post.dim() == 3 and y_post.shape[-1] == 6:
            valid = y_post[0][:, 4] > CONF_THRESH
            bboxes = y_post[0][valid].cpu().numpy()
        else:
            preds = non_max_suppression(y_post, conf_thres=CONF_THRESH, iou_thres=0.45, max_det=100)
            bboxes = preds[0].cpu().numpy()
    elif isinstance(det_out, torch.Tensor):
        preds = non_max_suppression(det_out, conf_thres=CONF_THRESH, iou_thres=0.45, max_det=100)
        bboxes = preds[0].cpu().numpy()
    else:
        bboxes = np.empty((0, 6))

    if len(bboxes):
        bboxes[:, [0, 2]] *= (orig_w / IMG_SIZE)
        bboxes[:, [1, 3]] *= (orig_h / IMG_SIZE)

    lane_prob = torch.sigmoid(out['lane_logits'])[0, 0].cpu().numpy()
    return bboxes, lane_prob

def draw_results(img_bgr, bboxes, lane_prob, lane_thresh=0.5):
    h, w = img_bgr.shape[:2]
    vis = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    lane_mask = cv2.resize(lane_prob, (w, h), interpolation=cv2.INTER_LINEAR)
    lane_binary = (lane_mask > lane_thresh).astype(np.uint8)
    overlay = vis.copy()
    overlay[lane_binary == 1] = (255, 0, 255)
    vis = cv2.addWeighted(vis, 0.65, overlay, 0.35, 0)

    for box in bboxes:
        x1, y1, x2, y2, conf, cls = box
        cls_id = int(cls)
        if cls_id >= NUM_VEHICLE_CLASSES:
            continue
        color = BOX_COLORS[cls_id]
        label = f"{VEHICLE_CLASSES[cls_id]} {conf:.2f}"
        cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(vis, (int(x1), int(y1)-th-4), (int(x1)+tw, int(y1)), color, -1)
        cv2.putText(vis, label, (int(x1), int(y1)-2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return vis

print("Inference functions ready.")

In [ ]:
import tarfile

# Extract dataset if needed
NB02_TAR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_yolo_nb02.tar")
if not os.path.isdir(os.path.join(DATASET_DIR, "images", "val")):
    assert os.path.isfile(NB02_TAR), f"NB02 tar not found: {NB02_TAR}"
    os.makedirs(DATASET_DIR, exist_ok=True)
    with tarfile.open(NB02_TAR, "r") as tar:
        tar.extractall(DATASET_DIR, filter='data')
    print("Dataset extracted.")
else:
    print("Dataset already present.")

# Select sample images
val_dir = os.path.join(DATASET_DIR, "images", "val")
all_images = sorted(os.listdir(val_dir))
random.seed(42)
sample_images = random.sample(all_images, min(6, len(all_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for i, fname in enumerate(sample_images):
    img_path = os.path.join(val_dir, fname)
    img_bgr = cv2.imread(img_path)
    bboxes, lane_prob = run_joint_inference(model, img_bgr)
    vis = draw_results(img_bgr, bboxes, lane_prob)

    ax = axes[i // 3, i % 3]
    ax.imshow(vis)
    ax.set_title(f"{fname} ({len(bboxes)} dets)")
    ax.axis('off')

plt.suptitle("Joint Inference: Vehicle Detection + Lane Segmentation", fontsize=14)
plt.tight_layout()
plt.show()

## 4 — Full Evaluation

In [ ]:
# Extract masks if needed
MASKS_TAR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_lane_masks.tar")
if not os.path.isdir(os.path.join(DATASET_DIR, "masks", "val")):
    assert os.path.isfile(MASKS_TAR), f"Masks tar not found: {MASKS_TAR}"
    with tarfile.open(MASKS_TAR, "r") as tar:
        tar.extractall(DATASET_DIR, filter='data')
    print("Masks extracted.")

from src.data.transforms import JointTransform
from src.data.dataset import JointBDDDataset, joint_collate_fn
from torch.utils.data import DataLoader

val_transform = JointTransform(img_size=640, mask_height=160, mask_width=160, augment=False)
val_ds = JointBDDDataset(
    images_dir=os.path.join(DATASET_DIR, "images", "val"),
    labels_dir=os.path.join(DATASET_DIR, "labels", "val"),
    masks_dir=os.path.join(DATASET_DIR, "masks", "val"),
    transform=val_transform,
)
val_loader = DataLoader(
    val_ds, batch_size=16, shuffle=False, num_workers=4,
    collate_fn=joint_collate_fn, pin_memory=True,
)
print(f"Validation: {len(val_ds)} samples, {len(val_loader)} batches")

In [ ]:
from src.metrics.detection import DetectionMetrics
from src.metrics.lane import LaneMetrics
from src.utils.class_map import remap_targets_bdd_to_vehicle
from tqdm import tqdm

det_metrics = DetectionMetrics(device='cuda')
lane_metrics = LaneMetrics(thresholds=[0.3, 0.4, 0.5, 0.6, 0.7])

print(f"Evaluating on {len(val_ds)} samples...")

for batch in tqdm(val_loader, desc="Evaluating"):
    images = batch['images'].to('cuda')
    det_targets = batch['det_targets'].to('cuda')
    lane_masks = batch['lane_masks'].to('cuda')

    with torch.no_grad(), torch.amp.autocast('cuda'):
        outputs = model(images)

    # Lane metrics
    lane_metrics.update(outputs['lane_logits'], lane_masks)

    # Detection metrics
    det_out = outputs['det_output']

    # Handle YOLO26 end2end output format
    if isinstance(det_out, (tuple, list)) and len(det_out) == 2:
        y_post = det_out[0]
        if isinstance(y_post, torch.Tensor) and y_post.dim() == 3 and y_post.shape[-1] == 6:
            preds = [y_post[bi][y_post[bi][:, 4] > 0.001] for bi in range(y_post.shape[0])]
        else:
            preds = non_max_suppression(y_post, conf_thres=0.001, iou_thres=0.6, max_det=300)
    elif isinstance(det_out, torch.Tensor):
        preds = non_max_suppression(det_out, conf_thres=0.001, iou_thres=0.6, max_det=300)
    else:
        preds = [torch.empty((0, 6), device=images.device)] * images.shape[0]

    vehicle_targets = remap_targets_bdd_to_vehicle(det_targets)

    for bi in range(images.shape[0]):
        pred_boxes = preds[bi].cpu()
        if vehicle_targets.shape[0] > 0:
            img_mask = vehicle_targets[:, 0] == bi
            img_gt = vehicle_targets[img_mask]
            if len(img_gt) > 0:
                gt_cls = img_gt[:, 1].long().cpu()
                gt_xywh = img_gt[:, 2:6].cpu()
                _, _, h, w = images.shape
                gt_boxes = torch.zeros(len(gt_xywh), 4)
                gt_boxes[:, 0] = (gt_xywh[:, 0] - gt_xywh[:, 2] / 2) * w
                gt_boxes[:, 1] = (gt_xywh[:, 1] - gt_xywh[:, 3] / 2) * h
                gt_boxes[:, 2] = (gt_xywh[:, 0] + gt_xywh[:, 2] / 2) * w
                gt_boxes[:, 3] = (gt_xywh[:, 1] + gt_xywh[:, 3] / 2) * h
            else:
                gt_boxes = torch.empty((0, 4))
                gt_cls = torch.empty((0,), dtype=torch.int64)
        else:
            gt_boxes = torch.empty((0, 4))
            gt_cls = torch.empty((0,), dtype=torch.int64)

        det_metrics.update(pred_boxes, gt_boxes, gt_cls)

det_results = det_metrics.compute()
lane_results = lane_metrics.compute()

In [ ]:
# Print results
print("=" * 55)
print(" DETECTION METRICS")
print("=" * 55)
print(f"  mAP@50-95: {det_results.get('det_map50_95', 0)*100:.2f}%")
print(f"  mAP@50:    {det_results.get('det_map50', 0)*100:.2f}%")
print()
for cls_name in VEHICLE_CLASSES:
    ap = det_results.get(f'ap_{cls_name}', None)
    if ap is not None:
        print(f"  AP {cls_name:<12}: {ap*100:.2f}%")

print()
print("=" * 55)
print(" LANE METRICS")
print("=" * 55)
print(f"  mIoU:         {lane_results.get('lane_miou', 0)*100:.2f}%")
print(f"  F1:           {lane_results.get('lane_f1', 0)*100:.2f}%")
print(f"  Best thresh:  {lane_results.get('lane_best_thresh', 0.5)}")
print(f"  Best F1:      {lane_results.get('lane_best_f1', 0)*100:.2f}%")
print("=" * 55)

## 5 — Lane Threshold Analysis

In [ ]:
# Plot F1 and IoU vs threshold
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
f1_vals = [lane_results.get(f'lane_f1_{t}', 0) for t in thresholds]
iou_vals = [lane_results.get(f'lane_iou_{t}', 0) for t in thresholds]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, [v*100 for v in f1_vals], 'o-', label='F1')
ax.plot(thresholds, [v*100 for v in iou_vals], 's-', label='IoU')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score (%)')
ax.set_title('Lane Segmentation: F1 and IoU vs Threshold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6 — Summary

In [ ]:
print("=" * 60)
print(" JOINT MODEL EVALUATION SUMMARY (v4)")
print("=" * 60)
print(f"  Architecture:      v4 native nc=5 + transformer lane head")
print(f"  Classes:           {', '.join(VEHICLE_CLASSES)}")
print(f"  Checkpoint:        {os.path.basename(JOINT_WEIGHTS)}")
print(f"  Epoch:             {epoch}")
print(f"  Vehicle mAP@50-95: {det_results.get('det_map50_95', 0)*100:.2f}%")
print(f"  Vehicle mAP@50:    {det_results.get('det_map50', 0)*100:.2f}%")
print(f"  Lane mIoU:         {lane_results.get('lane_miou', 0)*100:.2f}%")
print(f"  Lane F1:           {lane_results.get('lane_f1', 0)*100:.2f}%")
print("=" * 60)